In [22]:
import pandas as pd
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_distances

In [23]:
data = pd.read_csv("data/features-annotated-artistless-with-lyrics-and-languages.csv")
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Nationality,Continent,Points (Total),lyrics,detected_languages,num_languages,language_confidences
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.0,0.054,0.401,0.000,0.145,puerto rico,latin-america,200,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.0,0.086,0.468,0.000,0.675,united states,anglo-america,200,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.0,0.033,0.483,0.000,0.834,mexico,latin-america,200,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.0,0.038,0.222,0.003,0.536,united states,anglo-america,200,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.0,0.029,0.037,0.000,0.750,canada,anglo-america,200,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9156,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.0,0.127,0.160,0.001,0.587,brazil,latin-america,1,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0
9157,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.0,0.262,0.050,0.000,0.587,united states,anglo-america,1,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0
9158,5EgYDyMDCi8X0zDydTF7Rb,200,heartbeat (bts world origil soundtrack),bts,NaN,03/07/2019,0.437,0.716,-4306.0,0.050,0.390,0.000,0.307,south korea,asia,1,NaN,NaN,0,NaN
9159,01vv2AjxgP4uUyb8waYO5Y,200,morph,twenty one pilots,NaN,16/10/2018,0.734,0.607,-7249.0,0.081,0.075,0.001,0.518,united states,anglo-america,1,Can't stop thinking about if and when I die\nF...,ENGLISH,1,ENGLISH: 1.0


In [24]:
seed = np.random.randint(1000)
sample = data.sample(n=10, random_state=seed)

print(f"Seed = {seed}")
sample[["Title", "Primary Artists", "Featuring Artists"]]

Seed = 157


,Title,Primary Artists,Featuring Artists
6556,bonnie & clyde,"loreda, mozzik",NaN
9105,ni**as in paris,"jay-z, kanye west",NaN
2583,mirror,kendrick lamar,NaN
3559,old fashioned,panic! at the disco,NaN
175,more than you know,axwell /\ ingrosso,NaN
5187,seeing green (with drake & lil wayne),nicki minaj,"drake, lil wayne"
6252,wish wish (feat. cardi b & 21 savage),dj khaled,"21 savage, cardi b"
8855,mirrorball,taylor swift,NaN
7433,24,kanye west,NaN
868,ok,"james blunt, robin schulz",NaN


In [ ]:
features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

def get_recomendations(song_list, n_recs):
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Similarity'])

    # impute & scale using full dataset for consistency
    impute_vals = data[features].median()
    X_full_imputed = data[features].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=features)

    liked_idx = song_list.index
    liked_vectors = X_scaled.loc[liked_idx].values  # shape (m_liked, n_features)

    # pairwise distances between each liked song and every song in dataset
    dists = cosine_distances(liked_vectors, X_scaled.values)  # shape (m_liked, n_songs)
    sims_matrix = 1 - dists  # convert distances to similarities

    # aggregate similarities across liked songs
    sims = np.median(sims_matrix, axis=0)
    
    scores = pd.DataFrame({'Similarity': sims}, index=X_scaled.index)
    top = scores.drop(index=liked_idx).sort_values('Similarity', ascending=False).head(n_recs)

    recommendations = data.loc[top.index, ['Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
    recommendations['Similarity'] = top['Similarity']
    return recommendations

In [39]:
mask = data['Primary Artists'].fillna('').str.contains(r'\b(?:billie eilish)\b', case=False, regex=True)
liked_songs = data[mask].copy()

if liked_songs.shape[0] > 10:
    liked_songs = liked_songs.sample(10)

liked_songs[["Title", "Primary Artists", "Featuring Artists"] + features]

,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence
7614,bury a friend,billie eilish,NaN,0.905,0.389,-14505.0,0.332,0.740,0.162,0.196
6924,bad guy,billie eilish,NaN,0.701,0.425,-10965.0,0.375,0.328,0.130,0.562
4689,getting older,billie eilish,NaN,0.599,0.128,-17261.0,0.109,0.974,0.121,0.367
506,nda,billie eilish,NaN,0.765,0.373,-9915.0,0.071,0.329,0.541,0.590
3958,you should see me in a crown,billie eilish,NaN,0.697,0.552,-10271.0,0.196,0.479,0.127,0.278
8674,therefore i am,billie eilish,NaN,0.879,0.365,-7781.0,0.092,0.227,0.135,0.663
6965,wish you were gay,billie eilish,NaN,0.853,0.351,-10198.0,0.241,0.390,0.000,0.282
177,bury a friend,billie eilish,NaN,0.905,0.389,-14505.0,0.332,0.740,0.162,0.196
4514,bored,billie eilish,NaN,0.614,0.318,-12695.0,0.048,0.896,0.002,0.112
1335,idontwanbeyouanymore,billie eilish,NaN,0.483,0.412,-8461.0,0.040,0.737,0.000,0.247


In [ ]:
recomendations = get_recomendations(liked_songs, 10)
recomendations

,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Similarity
6269,surf,mac miller,NaN,0.828,0.182,-13066.0,0.166,0.754,0.071,0.231,0.836930
6396,revenge,xxxtentacion,NaN,0.746,0.251,-16169.0,0.259,0.782,0.002,0.180,0.828905
7808,"bad girls, good vibes",ufo361,NaN,0.826,0.368,-12761.0,0.133,0.516,0.025,0.158,0.806223
3124,goldwing,billie eilish,NaN,0.654,0.303,-13943.0,0.126,0.755,0.159,0.168,0.799700
5705,heart to heart,mac demarco,NaN,0.901,0.136,-16127.0,0.102,0.667,0.354,0.643,0.793121
2011,vérité,ninho,NaN,0.721,0.509,-9356.0,0.153,0.423,0.000,0.317,0.792676
6929,down,marian hill,NaN,0.593,0.362,-14701.0,0.273,0.684,0.003,0.052,0.792072
7027,change,j. cole,NaN,0.740,0.523,-10753.0,0.233,0.666,0.001,0.235,0.784485
6090,touch eem body,"bass santa, kin$oul, reddz, xxxtentacion",NaN,0.726,0.419,-11895.0,0.250,0.574,0.000,0.064,0.779678
8879,where this flower blooms (feat. frank ocean),"the creator, tyler",frank ocean,0.686,0.429,-10402.0,0.150,0.710,0.000,0.270,0.778707
